# K-Nearest Neighbors

**Companion lesson:** https://ml-viz.vercel.app/courses/knn-decision-trees/01-knn

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.facecolor'] = '#0f1117'
plt.rcParams['axes.facecolor'] = '#1a1d27'
plt.rcParams['text.color'] = 'white'
plt.rcParams['axes.labelcolor'] = '#94a3b8'
plt.rcParams['xtick.color'] = '#94a3b8'
plt.rcParams['ytick.color'] = '#94a3b8'
plt.rcParams['axes.edgecolor'] = '#2e3347'

## KNN from scratch

Classify by majority vote of the k nearest neighbors.

In [ ]:
class KNN:
    def __init__(self, k=3):
        self.k = k
    def fit(self, X, y):
        self.X_train = X
        self.y_train = y
    def predict(self, X):
        preds = []
        for x in X:
            dists = np.linalg.norm(self.X_train - x, axis=1)
            idx = np.argsort(dists)[:self.k]
            votes = self.y_train[idx]
            preds.append(np.bincount(votes).argmax())
        return np.array(preds)

np.random.seed(42)
n = 60
X0 = np.random.randn(n // 2, 2) + [-1.5, 0]
X1 = np.random.randn(n // 2, 2) + [1.5, 0]
X = np.vstack([X0, X1])
y = np.array([0] * (n // 2) + [1] * (n // 2))

knn = KNN(k=5)
knn.fit(X, y)

xx, yy = np.meshgrid(np.linspace(-5, 5, 100), np.linspace(-5, 5, 100))
grid = np.c_[xx.ravel(), yy.ravel()]
Z = knn.predict(grid).reshape(xx.shape)

fig, ax = plt.subplots(figsize=(8, 6))
ax.contourf(xx, yy, Z, alpha=0.2, cmap='RdYlBu')
ax.scatter(X[y == 0, 0], X[y == 0, 1], c='#f43f5e', s=20, alpha=0.7, label='Class 0')
ax.scatter(X[y == 1, 0], X[y == 1, 1], c='#818cf8', s=20, alpha=0.7, label='Class 1')
ax.legend()
ax.set_title('KNN Decision Boundary (k=5)', color='white')
plt.tight_layout()
plt.show()

## Effect of k on decision boundary

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 4))
for ax, k in zip(axes, [1, 3, 7, 20]):
    knn = KNN(k=k)
    knn.fit(X, y)
    Z = knn.predict(grid).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.2, cmap='RdYlBu')
    ax.scatter(X[y == 0, 0], X[y == 0, 1], c='#f43f5e', s=10, alpha=0.5)
    ax.scatter(X[y == 1, 0], X[y == 1, 1], c='#818cf8', s=10, alpha=0.5)
    ax.set_title(f'k = {k}', color='white')
    ax.set_xlim(-5, 5); ax.set_ylim(-5, 5)
plt.suptitle('KNN: Effect of k on Decision Boundary', color='white', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

## Distance metrics matter

Euclidean (L2) vs Manhattan (L1) change which neighbors are 'closest'.

In [ ]:
a = np.array([0, 0]); b = np.array([3, 4])
print('Euclidean (L2):', np.linalg.norm(a - b))
print('Manhattan (L1):', np.abs(a - b).sum())

## The curse of dimensionality

In high dimensions, distances concentrate — the nearest and farthest points become almost equally far, so 'nearest neighbor' loses meaning.

In [ ]:
rng = np.random.default_rng(0)
for d in [2, 10, 100, 1000]:
    X = rng.random((500, d))
    dists = np.linalg.norm(X - X[0], axis=1)[1:]
    ratio = dists.max() / dists.min()
    print(f'dim={d:>4}: (max/min distance) = {ratio:.2f}')

## Key takeaways

- KNN is **lazy**: no training — it stores the data and votes among the $k$ nearest points at query time.
- Small $k$ = flexible/noisy; large $k$ = smooth/biased.
- The **distance metric** and **feature scaling** strongly affect results — always standardize.
- It degrades in high dimensions (curse of dimensionality) and is slow at prediction time.